In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = spark.read.table("bankingdata.gold.fact_transaction")

df = df.withColumn("event_time", col("event_time").cast("timestamp"))


In [0]:
required_cols = [
    "transaction_id","account_id","customer_id","branch_id","card_id",
    "product_id","transaction_type","amount","merchant_category",
    "location","event_time","is_high_value"
]

### 1. Fraud Risk Alert

High value OR rapid transaction

In [0]:
w = Window.partitionBy("account_id").orderBy("event_time")

df1 = df.withColumn("prev_time", lag("event_time").over(w)) \
    .withColumn("prev_amount", lag("amount").over(w)) \
    .withColumn("prev_location", lag("location").over(w)) \
    .withColumn(
        "time_diff_sec",
        col("event_time").cast("long") - col("prev_time").cast("long")
    )

fraud_risk_df = df1.withColumn(
    "fraud_reason",
    when(
        (col("amount") > 100000),
        "VERY_HIGH_AMOUNT"
    ).when(
        (col("time_diff_sec") < 30) &
        (col("amount") > 5000),
        "RAPID_HIGH_VALUE_TXN"
    ).when(
        (col("time_diff_sec") < 300) &
        (col("location") != col("prev_location")) &
        (col("amount") > 1000),
        "LOCATION_CHANGE_SHORT_TIME"
    ).when(
        (col("is_high_value") == True) &
        (col("amount") > 5000) &
        (col("time_diff_sec") < 120),
        "HIGH_VALUE_FREQUENCY"
    )
)

fraud_risk_df = fraud_risk_df.filter(col("fraud_reason").isNotNull()) \
    .select(*required_cols, "fraud_reason", "time_diff_sec")

In [0]:
fraud_risk_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.businessinsights.fraud_risk")

### 2. Transaction Velocity Spike

Too many transactions in short time (5 min window)

In [0]:
velocity_df = df.groupBy(
    "account_id",
    window(col("event_time"), "5 minutes")
).agg(
    count("*").alias("txn_count")
)

velocity_risk_accounts = velocity_df.filter(
    col("txn_count") > 8
).select("account_id")

velocity_risk_df = df.join(
    velocity_risk_accounts,
    "account_id",
    "inner"
).select(*required_cols)

In [0]:
velocity_risk_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.businessinsights.velocity_risk")

### 3. Location Anomaly

Location changes within short time

In [0]:
w = Window.partitionBy("account_id").orderBy("event_time")

df2 = df.withColumn("prev_location", lag("location").over(w)) \
    .withColumn("prev_time", lag("event_time").over(w)) \
    .withColumn("time_diff_min",
        (col("event_time").cast("long") - col("prev_time").cast("long")) / 60
    )

location_risk_df = df2.filter(
    (col("location") != col("prev_location")) &
    (col("time_diff_min") < 10)
).select(*required_cols)

In [0]:
location_risk_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.businessinsights.location_risk")